In [1]:
import requests
import pandas as pd
from getpass import getpass

api_key = getpass("Enter your Census API key: ") 


Enter your Census API key:  ········


In [2]:
print("API key loaded:", bool(api_key))
print("Key length:", len(api_key))

API key loaded: True
Key length: 40


In [3]:
import requests
import pandas as pd

url = "https://api.census.gov/data/2023/acs/acs5"

counties = {
    "005": "Bronx",
    "047": "Brooklyn",
    "061": "Manhattan",
    "081": "Queens",
    "085": "Staten Island"
}

columns = [f"B01001_{i:03d}E" for i in range(1, 50)]

dataframes = []

for county, borough in counties.items():

    params = {
        "get": ",".join(columns),
        "for": "tract:*",
        "in": f"state:36 county:{county}",
        "key": api_key
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    data = response.json()

    df = pd.DataFrame(data[1:], columns=data[0])
    df["Borough"] = borough

    dataframes.append(df)

    print("Successfully retrieved:", borough)

age_population = pd.concat(dataframes, ignore_index=True)

# Create census tract identifiers
age_population["GEOID"] = (
    age_population["state"] +
    age_population["county"] +
    age_population["tract"]
)

# Convert population counts to numeric values
age_population[columns] = age_population[columns].apply(
    pd.to_numeric, errors="coerce"
)

age_population.head()

Successfully retrieved: Bronx
Successfully retrieved: Brooklyn
Successfully retrieved: Manhattan
Successfully retrieved: Queens
Successfully retrieved: Staten Island


,B01001_001E,B01001_002E,B01001_003E,B01001_004E,B01001_005E,B01001_006E,B01001_007E,B01001_008E,B01001_009E,B01001_010E,...,B01001_045E,B01001_046E,B01001_047E,B01001_048E,B01001_049E,state,county,tract,Borough,GEOID
0,3538,3259,0,0,0,0,82,59,83,153,...,0,0,0,0,0,36,005,000100,Bronx,36005000100
1,5177,2647,72,72,185,127,53,0,137,138,...,120,59,74,83,86,36,005,000200,Bronx,36005000200
2,6481,3104,39,20,328,86,39,39,50,99,...,185,27,34,0,58,36,005,000400,Bronx,36005000400
3,6011,2676,193,153,123,160,130,161,64,111,...,173,131,232,121,38,36,005,001600,Bronx,36005001600
4,2401,1216,77,44,58,58,3,0,33,0,...,0,31,0,0,0,36,005,001901,Bronx,36005001901


In [4]:
age_groups = {
    "Under_5": [3, 27],
    "5_9": [4, 28],
    "10_14": [5, 29],
    "15_17": [6, 30],
    "18_19": [7, 31],
    "20": [8, 32],
    "21": [9, 33],
    "22_24": [10, 34],
    "25_29": [11, 35],
    "30_34": [12, 36],
    "35_39": [13, 37],
    "40_44": [14, 38],
    "45_49": [15, 39],
    "50_54": [16, 40],
    "55_59": [17, 41],
    "60_61": [18, 42],
    "62_64": [19, 43],
    "65_66": [20, 44],
    "67_69": [21, 45],
    "70_74": [22, 46],
    "75_79": [23, 47],
    "80_84": [24, 48],
    "85_and_Over": [25, 49]
}

for age_group, codes in age_groups.items():

    census_columns = [
        f"B01001_{code:03d}E" for code in codes
    ]

    age_population[age_group] = age_population[
        census_columns
    ].sum(axis=1, min_count=2)

age_population["Total_population"] = age_population["B01001_001E"]

# Keep only relevant columns
age_population = age_population[
    ["GEOID", "Borough", "Total_population"] +
    list(age_groups.keys())
].copy()

age_population.head()

,GEOID,Borough,Total_population,Under_5,5_9,10_14,15_17,18_19,20,21,...,50_54,55_59,60_61,62_64,65_66,67_69,70_74,75_79,80_84,85_and_Over
0,36005000100,Bronx,3538,0,0,0,0,82,59,97,...,251,128,27,30,6,26,5,7,39,0
1,36005000200,Bronx,5177,261,140,343,138,182,14,149,...,244,180,153,521,89,171,96,137,136,86
2,36005000400,Bronx,6481,256,351,585,254,59,50,50,...,483,331,77,300,60,497,121,62,39,86
3,36005001600,Bronx,6011,232,265,441,328,269,201,126,...,433,424,180,132,91,353,271,276,136,116
4,36005001901,Bronx,2401,208,63,207,78,4,1,46,...,203,92,37,36,0,0,31,0,4,0


In [7]:
age_population.to_csv(
    "NYCAgePopulation2023.csv",
    index=False
)

In [11]:
crosswalk_url = (
    "https://data.cityofnewyork.us/api/views/"
    "hm78-6dwm/rows.csv?accessType=DOWNLOAD"
)

crosswalk = pd.read_csv(crosswalk_url, dtype=str)

crosswalk.columns = crosswalk.columns.str.strip().str.lower()

print(crosswalk.columns.tolist())

crosswalk.head()

['geoid', 'countyfips', 'borocode', 'boroname', 'boroct2020', 'ct2020', 'ctlabel', 'ntacode', 'ntatype', 'ntaname', 'ntaabbrev', 'cdtacode', 'cdtatype', 'cdtaname']


,geoid,countyfips,borocode,boroname,boroct2020,ct2020,ctlabel,ntacode,ntatype,ntaname,ntaabbrev,cdtacode,cdtatype,cdtaname
0,36005001901,005,2,Bronx,2001901,001901,19.01,BX0101,0,Mott Haven-Port Morris,MttHvn,BX01,CD,BX01 Melrose-Mott Haven-Port Morris (CD 1 Appr...
1,36005001902,005,2,Bronx,2001902,001902,19.02,BX0101,0,Mott Haven-Port Morris,MttHvn,BX01,CD,BX01 Melrose-Mott Haven-Port Morris (CD 1 Appr...
2,36005001903,005,2,Bronx,2001903,001903,19.03,BX0101,0,Mott Haven-Port Morris,MttHvn,BX01,CD,BX01 Melrose-Mott Haven-Port Morris (CD 1 Appr...
3,36005002300,005,2,Bronx,2002300,002300,23,BX0101,0,Mott Haven-Port Morris,MttHvn,BX01,CD,BX01 Melrose-Mott Haven-Port Morris (CD 1 Appr...
4,36005002500,005,2,Bronx,2002500,002500,25,BX0101,0,Mott Haven-Port Morris,MttHvn,BX01,CD,BX01 Melrose-Mott Haven-Port Morris (CD 1 Appr...


In [12]:
print("Census API GEOIDs:")
print(age_population["GEOID"].head(10))

print("Crosswalk GEOIDs:")
print(crosswalk["geoid"].head(10))

Census API GEOIDs:
0    36005000100
1    36005000200
2    36005000400
3    36005001600
4    36005001901
5    36005001902
6    36005001903
7    36005001904
8    36005002001
9    36005002002
Name: GEOID, dtype: str
Crosswalk GEOIDs:
0    36005001901
1    36005001902
2    36005001903
3    36005002300
4    36005002500
5    36005002701
6    36005002702
7    36005003100
8    36005003300
9    36005003500
Name: geoid, dtype: str


In [13]:
print("Census GEOID lengths:")
print(age_population["GEOID"].str.len().value_counts())

print("Crosswalk GEOID lengths:")
print(crosswalk["geoid"].str.len().value_counts())

Census GEOID lengths:
GEOID
11    2327
Name: count, dtype: int64
Crosswalk GEOID lengths:
geoid
11    2327
Name: count, dtype: int64


In [14]:
print("Census GEOID lengths:")
print(age_population["GEOID"].str.len().value_counts())

print("Crosswalk GEOID lengths:")
print(crosswalk["geoid"].str.len().value_counts())

Census GEOID lengths:
GEOID
11    2327
Name: count, dtype: int64
Crosswalk GEOID lengths:
geoid
11    2327
Name: count, dtype: int64


In [15]:
print(
    "Duplicate Census GEOIDs:",
    age_population["GEOID"].duplicated().sum()
)

print(
    "Duplicate crosswalk GEOIDs:",
    crosswalk["geoid"].duplicated().sum()
)

Duplicate Census GEOIDs: 0
Duplicate crosswalk GEOIDs: 0


In [18]:
age_population["GEOID"] = (
    age_population["GEOID"].astype(str).str.strip()
)

crosswalk["geoid"] = (
    crosswalk["geoid"].astype(str).str.strip()
)

population_merged = age_population.merge(
    crosswalk,
    left_on="GEOID",
    right_on="geoid",
    how="left",
    validate="one_to_one",
    indicator=True
)

print(population_merged["_merge"].value_counts())

_merge
both          2327
left_only        0
right_only       0
Name: count, dtype: int64


In [19]:
print(population_merged["cdtatype"].value_counts())

print(
    population_merged[
        ["cdtacode", "cdtaname", "cdtatype"]
    ]
    .drop_duplicates()
    .sort_values("cdtacode")
    .to_string(index=False)
)

cdtatype
CD     2305
JIA      22
Name: count, dtype: int64
cdtacode                                                        cdtaname cdtatype
    BK01                  BK01 Williamsburg-Greenpoint (CD 1 Equivalent)       CD
    BK02         BK02 Downtown Brooklyn-Fort Greene (CD 2 Approximation)       CD
    BK03                    BK03 Bedford-Stuyvesant (CD 3 Approximation)       CD
    BK04                                 BK04 Bushwick (CD 4 Equivalent)       CD
    BK05           BK05 East New York-Cypress Hills (CD 5 Approximation)       CD
    BK06            BK06 Park Slope-Carroll Gardens (CD 6 Approximation)       CD
    BK07           BK07 Sunset Park-Windsor Terrace (CD 7 Approximation)       CD
    BK08                 BK08 Crown Heights (North) (CD 8 Approximation)       CD
    BK09                 BK09 Crown Heights (South) (CD 9 Approximation)       CD
    BK10              BK10 Bay Ridge-Dyker Heights (CD 10 Approximation)       CD
    BK11               BK11 Bensonhurst

In [20]:
population_columns = [
    "Total_population"
] + list(age_groups.keys())

community_age_population = (
    population_merged.groupby(
        ["cdtacode", "cdtaname", "cdtatype"],
        as_index=False
    )[population_columns]
    .sum(min_count=1)
)

community_age_population.head()

,cdtacode,cdtaname,cdtatype,Total_population,Under_5,5_9,10_14,15_17,18_19,20,...,50_54,55_59,60_61,62_64,65_66,67_69,70_74,75_79,80_84,85_and_Over
0,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),CD,203002,17411,11959,11893,6710,3583,1328,...,7828,7704,2459,4667,2096,3446,5546,2956,2140,2432
1,BK02,BK02 Downtown Brooklyn-Fort Greene (CD 2 Appro...,CD,125637,8121,4828,4635,2236,3309,1206,...,7241,5843,2250,2346,1521,2734,3957,3085,2197,2232
2,BK03,BK03 Bedford-Stuyvesant (CD 3 Approximation),CD,180283,12121,10654,11389,6514,2825,1554,...,9563,8709,3548,4072,2244,3352,5210,3346,2362,1981
3,BK04,BK04 Bushwick (CD 4 Equivalent),CD,111975,4157,4346,5216,2939,1735,1273,...,5459,4965,1714,2628,1968,1888,3004,2193,1245,1316
4,BK05,BK05 East New York-Cypress Hills (CD 5 Approxi...,CD,201036,13118,13045,15109,8194,5383,2427,...,12959,13342,5409,6870,4441,5716,6793,5101,3318,4030


In [21]:
community_age_population["Age_Total_Check"] = (
    community_age_population[list(age_groups.keys())]
    .sum(axis=1, min_count=len(age_groups))
)

community_age_population["Difference"] = (
    community_age_population["Total_population"]
    - community_age_population["Age_Total_Check"]
)

print(community_age_population["Difference"].value_counts())

Difference
0    71
Name: count, dtype: int64


In [22]:
print("Number of geographic areas:", len(community_age_population))

print(community_age_population.columns.tolist())

community_age_population.head(10)

Number of geographic areas: 71
['cdtacode', 'cdtaname', 'cdtatype', 'Total_population', 'Under_5', '5_9', '10_14', '15_17', '18_19', '20', '21', '22_24', '25_29', '30_34', '35_39', '40_44', '45_49', '50_54', '55_59', '60_61', '62_64', '65_66', '67_69', '70_74', '75_79', '80_84', '85_and_Over', 'Age_Total_Check', 'Difference']


,cdtacode,cdtaname,cdtatype,Total_population,Under_5,5_9,10_14,15_17,18_19,20,...,60_61,62_64,65_66,67_69,70_74,75_79,80_84,85_and_Over,Age_Total_Check,Difference
0,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),CD,203002,17411,11959,11893,6710,3583,1328,...,2459,4667,2096,3446,5546,2956,2140,2432,203002,0
1,BK02,BK02 Downtown Brooklyn-Fort Greene (CD 2 Appro...,CD,125637,8121,4828,4635,2236,3309,1206,...,2250,2346,1521,2734,3957,3085,2197,2232,125637,0
2,BK03,BK03 Bedford-Stuyvesant (CD 3 Approximation),CD,180283,12121,10654,11389,6514,2825,1554,...,3548,4072,2244,3352,5210,3346,2362,1981,180283,0
3,BK04,BK04 Bushwick (CD 4 Equivalent),CD,111975,4157,4346,5216,2939,1735,1273,...,1714,2628,1968,1888,3004,2193,1245,1316,111975,0
4,BK05,BK05 East New York-Cypress Hills (CD 5 Approxi...,CD,201036,13118,13045,15109,8194,5383,2427,...,5409,6870,4441,5716,6793,5101,3318,4030,201036,0
5,BK06,BK06 Park Slope-Carroll Gardens (CD 6 Approxim...,CD,121009,8956,7135,6470,3636,890,757,...,2207,2843,2297,2630,4070,3445,1251,1068,121009,0
6,BK07,BK07 Sunset Park-Windsor Terrace (CD 7 Approxi...,CD,120253,6752,8289,9635,3904,1894,888,...,2475,3240,2407,2910,3927,2623,1855,1564,120253,0
7,BK08,BK08 Crown Heights (North) (CD 8 Approximation),CD,109719,6878,4761,5332,2698,927,509,...,2100,2808,2005,2455,4245,1877,1895,1335,109719,0
8,BK09,BK09 Crown Heights (South) (CD 9 Approximation),CD,97675,5808,4870,5336,3311,1950,1374,...,2154,3376,2008,3052,3890,2705,2360,1973,97675,0
9,BK10,BK10 Bay Ridge-Dyker Heights (CD 10 Approximat...,CD,125053,6527,7470,6899,4726,2278,781,...,3178,4498,3020,4084,6549,4804,2486,3000,125053,0


In [24]:
community_age_population.to_csv(
    "CDTAAgePopulation2023.csv",
    index=False
)

In [25]:
district_population = population_merged[
    population_merged["cdtatype"] == "CD"
].copy()

print("Number of census tracts:", len(district_population))

print(
    "Number of community districts:",
    district_population["cdtacode"].nunique()
)

Number of census tracts: 2305
Number of community districts: 59


In [26]:
borough_codes = {
    "MN": "1",
    "BX": "2",
    "BK": "3",
    "QN": "4",
    "SI": "5"
}

district_population["Community_District"] = (
    district_population["cdtacode"].str[:2].map(borough_codes)
    + district_population["cdtacode"].str[2:]
).astype(int)

district_population[
    ["cdtacode", "Community_District", "cdtaname"]
].head()

,cdtacode,Community_District,cdtaname
0,QN01,401,QN01 Astoria-Queensbridge (CD 1 Equivalent)
1,BX09,209,BX09 Soundview-Parkchester (CD 9 Approximation)
2,BX09,209,BX09 Soundview-Parkchester (CD 9 Approximation)
3,BX09,209,BX09 Soundview-Parkchester (CD 9 Approximation)
4,BX01,201,BX01 Melrose-Mott Haven-Port Morris (CD 1 Appr...


In [27]:
population_columns = [
    "Total_population"
] + list(age_groups.keys())

community_age_population = (
    district_population.groupby(
        ["Community_District", "cdtacode", "cdtaname"],
        as_index=False
    )[population_columns]
    .sum(min_count=1)
)

community_age_population.head()

,Community_District,cdtacode,cdtaname,Total_population,Under_5,5_9,10_14,15_17,18_19,20,...,50_54,55_59,60_61,62_64,65_66,67_69,70_74,75_79,80_84,85_and_Over
0,101,MN01,MN01 Financial District-Tribeca (CD 1 Equivalent),71865,3466,3090,2653,1411,2141,1049,...,4207,3368,1368,1189,1050,1348,1224,1040,1138,1048
1,102,MN02,MN02 Greenwich Village-SoHo (CD 2 Equivalent),86202,2375,1847,2378,951,3650,908,...,5543,4632,1511,2524,2101,2872,3859,3244,1540,1342
2,103,MN03,MN03 Lower East Side-Chinatown (CD 3 Equivalent),153288,4316,4752,5320,2849,5223,2268,...,8185,11090,3393,4978,3737,5009,8495,4739,3710,4681
3,104,MN04,MN04 Chelsea-Hell's Kitchen (CD 4 Approximation),117099,3151,2830,2256,1559,1059,868,...,7159,8670,3283,3494,2602,3482,5918,3321,2212,1696
4,105,MN05,MN05 Midtown-Flatiron-Union Square (CD 5 Appro...,51619,1273,1246,955,617,1813,587,...,3086,2672,1609,1210,1117,1307,2717,1512,715,1077


In [28]:
print("Number of community districts:",
      community_age_population["Community_District"].nunique())

print("Missing values:")
print(community_age_population.isna().sum())

print("Duplicate community districts:",
      community_age_population["Community_District"].duplicated().sum())

Number of community districts: 59
Missing values:
Community_District    0
cdtacode              0
cdtaname              0
Total_population      0
Under_5               0
5_9                   0
10_14                 0
15_17                 0
18_19                 0
20                    0
21                    0
22_24                 0
25_29                 0
30_34                 0
35_39                 0
40_44                 0
45_49                 0
50_54                 0
55_59                 0
60_61                 0
62_64                 0
65_66                 0
67_69                 0
70_74                 0
75_79                 0
80_84                 0
85_and_Over           0
dtype: int64
Duplicate community districts: 0


In [29]:
age_total = community_age_population[
    list(age_groups.keys())
].sum(axis=1, min_count=len(age_groups))

difference = (
    community_age_population["Total_population"] - age_total
)

print(difference.value_counts())

0    59
Name: count, dtype: int64


In [30]:
print(
    "Negative population values:",
    (community_age_population[population_columns] < 0).sum().sum()
)

Negative population values: 0


In [31]:
community_age_population.to_csv(
    "CommunityDistrictAgePopulation2023.csv",
    index=False
)

In [33]:
print(community_age_population.columns.tolist())

['Community_District', 'cdtacode', 'cdtaname', 'Total_population', 'Under_5', '5_9', '10_14', '15_17', '18_19', '20', '21', '22_24', '25_29', '30_34', '35_39', '40_44', '45_49', '50_54', '55_59', '60_61', '62_64', '65_66', '67_69', '70_74', '75_79', '80_84', '85_and_Over']


In [34]:
community_age_population.head()

,Community_District,cdtacode,cdtaname,Total_population,Under_5,5_9,10_14,15_17,18_19,20,...,50_54,55_59,60_61,62_64,65_66,67_69,70_74,75_79,80_84,85_and_Over
0,101,MN01,MN01 Financial District-Tribeca (CD 1 Equivalent),71865,3466,3090,2653,1411,2141,1049,...,4207,3368,1368,1189,1050,1348,1224,1040,1138,1048
1,102,MN02,MN02 Greenwich Village-SoHo (CD 2 Equivalent),86202,2375,1847,2378,951,3650,908,...,5543,4632,1511,2524,2101,2872,3859,3244,1540,1342
2,103,MN03,MN03 Lower East Side-Chinatown (CD 3 Equivalent),153288,4316,4752,5320,2849,5223,2268,...,8185,11090,3393,4978,3737,5009,8495,4739,3710,4681
3,104,MN04,MN04 Chelsea-Hell's Kitchen (CD 4 Approximation),117099,3151,2830,2256,1559,1059,868,...,7159,8670,3283,3494,2602,3482,5918,3321,2212,1696
4,105,MN05,MN05 Midtown-Flatiron-Union Square (CD 5 Appro...,51619,1273,1246,955,617,1813,587,...,3086,2672,1609,1210,1117,1307,2717,1512,715,1077


In [35]:
community_age_population.to_csv(
    "CommunityDistrictAgePopulation2020.csv",
    index=False
)

In [36]:
import os

print(os.path.exists("CommunityDistrictAgePopulation2023.csv"))
print(community_age_population.shape)

True
(59, 27)


In [ ]:
"CombinedDeathRatesByCommunityDistrict2023.csv"